# Number Systems — Exercises

20 graded exercises covering every section of [notes.md](notes.md) and [theory.ipynb](theory.ipynb).

**Format**: Each exercise has:
- 🎯 **Problem** — real-world ML context
- 📝 **Scaffold** — fill in `# YOUR CODE HERE`
- ✅ **Solution** — in a separate cell, with assertions and explanations

| Level | Description | Exercises |
|-------|-------------|----------|
| ⭐ | Essential — core concepts | 1–6 |
| ⭐⭐ | Applied — real ML engineering | 7–14 |
| ⭐⭐⭐ | Interview/Production — debugging & optimization | 15–20 |

| Section | Exercises Covered |
|---------|-------------------|
| Positional Systems & Two's Complement | 1, 2 |
| IEEE 754 Internals | 3, 4 |
| AI Floating-Point Formats | 5, 6 |
| Quantization (INT8/INT4) | 7, 8, 9 |
| Non-Uniform Formats (NF4, Ternary) | 10 |
| Floating-Point Arithmetic | 11, 12 |
| Numerical Stability | 13, 14 |
| Quantization Mathematics (SQNR, Lloyd-Max) | 15, 16 |
| Mixed Precision & Memory | 17, 18 |
| Training Stability & Production | 19, 20 |

In [ ]:
import numpy as np
import struct
import warnings
from typing import Tuple, List

np.random.seed(42)
np.set_printoptions(precision=8, suppress=True)

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    torch.manual_seed(42)
    HAS_TORCH = True
    print(f'NumPy {np.__version__} | PyTorch {torch.__version__} — Ready!')
except ImportError:
    HAS_TORCH = False
    print(f'NumPy {np.__version__} — Ready! (PyTorch exercises will use NumPy fallbacks)')

---

## Exercise 1: Base Conversion Engine ⭐

You need to inspect raw weight bytes dumped from a GPU. Understanding binary and hex is essential.

**Task**: Implement `decimal_to_binary(n)` and `decimal_to_hex(n)` for *non-negative* integers.
Then implement `binary_to_decimal(bits_str)` to convert back.

**Requirements**:
- No use of Python's built-in `bin()`, `hex()`, or `int(x, base)`
- Handle `n = 0` correctly
- `binary_to_decimal` must compute the sum of positional values

In [ ]:
# Exercise 1: Your Solution

print('Scaffold placeholder: replace the code below with your implementation and rerun this cell.')
print('The original scaffold is preserved below inside an if False block.')

if False:
    
    def decimal_to_binary(n: int) -> str:
        """Convert non-negative integer to binary string (no '0b' prefix)."""
        # YOUR CODE HERE
        pass
    
    def decimal_to_hex(n: int) -> str:
        """Convert non-negative integer to hex string (no '0x' prefix)."""
        # YOUR CODE HERE
        pass
    
    def binary_to_decimal(bits: str) -> int:
        """Convert binary string to integer using positional summation."""
        # YOUR CODE HERE
        # Hint: sum d_k * 2^k for each bit position k
        pass
    
    # Tests
    for val in [0, 1, 42, 127, 255, 1024]:
        b = decimal_to_binary(val)
        h = decimal_to_hex(val)
        back = binary_to_decimal(b)
        print(f'{val:>6} → binary: {b:>12}  hex: {h:>4}  round-trip: {back}')
    pass


In [ ]:
# Exercise 1: Solution ✅

def decimal_to_binary(n: int) -> str:
    if n == 0:
        return '0'
    result = []
    while n > 0:
        result.append(str(n % 2))
        n //= 2
    return ''.join(reversed(result))

def decimal_to_hex(n: int) -> str:
    if n == 0:
        return '0'
    digits = '0123456789ABCDEF'
    result = []
    while n > 0:
        result.append(digits[n % 16])
        n //= 16
    return ''.join(reversed(result))

def binary_to_decimal(bits: str) -> int:
    total = 0
    for k, digit in enumerate(reversed(bits)):
        total += int(digit) * (2 ** k)
    return total

# Verify
for val in [0, 1, 42, 127, 255, 1024]:
    b = decimal_to_binary(val)
    h = decimal_to_hex(val)
    back = binary_to_decimal(b)
    assert back == val, f'Round-trip failed for {val}'
    print(f'{val:>6} → binary: {b:>12}  hex: {h:>4}  round-trip: {back} ✓')

print('\nAll tests passed ✓')
print('→ This is the basis for understanding how INT8/INT4 values are stored')

---

## Exercise 2: Two's Complement & Quantization Ranges ⭐

INT8 quantization uses two's complement to represent signed integers in [-128, 127].
INT4 uses [-8, 7]. Understanding the encoding is essential for debugging quantized models.

**Task**: Implement `twos_complement_encode(n, bits)` and `twos_complement_decode(binary, bits)` that:
- Encode a signed integer into two's complement binary
- Decode a two's complement binary string back to signed integer
- Handle the full range: for 8-bit, [-128, 127]

**Two's complement for negative n**: flip bits of |n|, add 1.

In [ ]:
# Exercise 2: Your Solution

print('Scaffold placeholder: replace the code below with your implementation and rerun this cell.')
print('The original scaffold is preserved below inside an if False block.')

if False:
    
    def twos_complement_encode(n: int, bits: int = 8) -> str:
        """Encode signed integer in two's complement."""
        # YOUR CODE HERE
        # For positive: just convert to binary, pad to `bits` width
        # For negative: flip bits of |n|, add 1
        pass
    
    def twos_complement_decode(binary: str, bits: int = 8) -> int:
        """Decode two's complement binary string to signed integer."""
        # YOUR CODE HERE
        # If MSB is 1: it's negative → flip bits, add 1, negate
        pass
    
    # Tests
    for val in [42, -42, 0, 127, -128, 1, -1, 7, -8]:
        bits = 8 if abs(val) > 7 or val == -128 else 4
        encoded = twos_complement_encode(val, bits)
        decoded = twos_complement_decode(encoded, bits)
        print(f'{val:>5} → {encoded} → {decoded}')
    pass


In [ ]:
# Exercise 2: Solution ✅

def twos_complement_encode(n: int, bits: int = 8) -> str:
    if n >= 0:
        return f'{n:0{bits}b}'
    else:
        # Flip bits of |n|, add 1
        pos = f'{abs(n):0{bits}b}'
        flipped = ''.join('1' if b == '0' else '0' for b in pos)
        result = int(flipped, 2) + 1
        return f'{result:0{bits}b}'

def twos_complement_decode(binary: str, bits: int = 8) -> int:
    if binary[0] == '0':  # positive
        return int(binary, 2)
    else:  # negative: flip, add 1, negate
        flipped = ''.join('1' if b == '0' else '0' for b in binary)
        return -(int(flipped, 2) + 1)

# Verify all INT8 edge cases
for val in [42, -42, 0, 127, -128, 1, -1]:
    enc = twos_complement_encode(val, 8)
    dec = twos_complement_decode(enc, 8)
    assert dec == val, f'Failed for {val}'
    print(f'INT8: {val:>5} → {enc} → {dec} ✓')

# Verify INT4 edge cases
print()
for val in [7, -8, 0, -1, 3, -3]:
    enc = twos_complement_encode(val, 4)
    dec = twos_complement_decode(enc, 4)
    assert dec == val
    print(f'INT4: {val:>5} → {enc} → {dec} ✓')

print(f'\n→ INT8 range: [{twos_complement_decode("10000000")}, {twos_complement_decode("01111111")}] = [-128, 127] → 256 levels')
print(f'→ INT4 range: [{twos_complement_decode("1000", 4)}, {twos_complement_decode("0111", 4)}] = [-8, 7] → 16 levels')

---

## Exercise 3: IEEE 754 Float Decoder ⭐

A training run produces NaN. You dump the raw bytes: `0x7FC00000`. What value is this?

**Task**: Implement `decode_fp32(value)` that returns a dict with:
- `sign`: 0 or 1
- `exponent_biased`: 0–255
- `exponent_true`: biased - 127
- `mantissa_bits`: 23-character string
- `mantissa_value`: fractional value of mantissa
- `category`: 'zero', 'subnormal', 'normal', 'infinity', 'NaN'
- `formula`: reconstructed value string

Use `struct.pack('>f', value)` to get raw bytes.

In [ ]:
# Exercise 3: Your Solution

print('Scaffold placeholder: replace the code below with your implementation and rerun this cell.')
print('The original scaffold is preserved below inside an if False block.')

if False:
    
    def decode_fp32(value: float) -> dict:
        """
        Decode IEEE 754 FP32 into components.
        Steps:
          1. struct.pack('>f', value) → 4 bytes
          2. Convert bytes to 32-bit binary string
          3. Split: sign(1) | exponent(8) | mantissa(23)
          4. Classify: zero, subnormal, normal, inf, NaN
        """
        # YOUR CODE HERE
        pass
    
    # Tests
    for v in [1.0, -13.625, 0.1, 0.0, float('inf'), float('nan')]:
        d = decode_fp32(v)
        print(f'{v:>10} → S={d["sign"]} E={d["exponent_biased"]:>3} '
              f'(true={d["exponent_true"]:>4}) M={d["mantissa_bits"]} [{d["category"]}]')
    pass


In [ ]:
# Exercise 3: Solution ✅

def decode_fp32(value: float) -> dict:
    packed = struct.pack('>f', value)
    bits = ''.join(f'{byte:08b}' for byte in packed)

    sign = int(bits[0])
    exp_bits = bits[1:9]
    man_bits = bits[9:]

    biased_exp = int(exp_bits, 2)
    true_exp = biased_exp - 127
    mantissa_val = sum(int(b) * 2**(-i-1) for i, b in enumerate(man_bits))

    if biased_exp == 0:
        category = 'zero' if mantissa_val == 0 else 'subnormal'
    elif biased_exp == 255:
        category = 'infinity' if mantissa_val == 0 else 'NaN'
    else:
        category = 'normal'

    if category == 'normal':
        formula = f'(-1)^{sign} × {1+mantissa_val:.6f} × 2^{true_exp}'
    else:
        formula = category

    return {
        'sign': sign, 'exponent_biased': biased_exp, 'exponent_true': true_exp,
        'mantissa_bits': man_bits, 'mantissa_value': mantissa_val,
        'category': category, 'formula': formula,
        'hex': packed.hex(), 'all_bits': f'{bits[0]}|{exp_bits}|{man_bits}'
    }

# Verify with the notes.md worked example: -13.625
d = decode_fp32(-13.625)
print('Worked example: -13.625')
print(f'  Bits: {d["all_bits"]}')
print(f'  Sign={d["sign"]}, Biased exp={d["exponent_biased"]}, True exp={d["exponent_true"]}')
print(f'  1+M = {1+d["mantissa_value"]:.6f}')
print(f'  Formula: {d["formula"]}')
print(f'  Hex: 0x{d["hex"]}')

# Verify special values
print(f'\nSpecial values:')
for v in [0.0, float('inf'), float('-inf'), float('nan')]:
    d = decode_fp32(v)
    print(f'  {str(v):>5} → 0x{d["hex"]}  [{d["category"]}]')

print('\nAll tests passed ✓')
print('→ 0x7FC00000 is the canonical quiet NaN on x86')
print('→ Use this decoder to inspect suspicious values during training')

---

## Exercise 4: Machine Epsilon & Precision Limits ⭐

Your BF16 training seems to stall — weight updates are applied but weights don't change.

**Task**:
- (a) Compute machine epsilon for FP16, FP32, FP64 using the iterative method:
  start with `eps = 1.0`, keep halving until `1.0 + eps == 1.0`
- (b) For BF16 (7 mantissa bits), compute epsilon as $2^{-7}$
- (c) If a weight is `1.0` and gradient is `0.001`, will the update be captured in BF16?
- (d) What is the minimum gradient magnitude that can update a BF16 weight of value `1.0`?

In [ ]:
# Exercise 4: Your Solution

print('Scaffold placeholder: replace the code below with your implementation and rerun this cell.')
print('The original scaffold is preserved below inside an if False block.')

if False:
    
    def compute_epsilon(dtype) -> float:
        """Find machine epsilon by halving until 1+eps rounds to 1."""
        # YOUR CODE HERE
        # eps = dtype(1.0)
        # while dtype(1.0) + dtype(eps/2) != dtype(1.0): ...
        pass
    
    # (a) Epsilon for each format
    for dtype in [np.float16, np.float32, np.float64]:
        eps = compute_epsilon(dtype)
        print(f'{dtype.__name__}: ε = {eps}')
    
    # (b) BF16 epsilon
    bf16_eps = None  # YOUR CODE HERE
    print(f'BF16: ε = {bf16_eps}')
    
    # (c) Will gradient 0.001 update weight 1.0 in BF16?
    # YOUR ANSWER HERE
    
    # (d) Minimum gradient for BF16 weight of 1.0
    # YOUR ANSWER HERE
    pass


In [ ]:
# Exercise 4: Solution ✅

def compute_epsilon(dtype) -> float:
    eps = dtype(1.0)
    while dtype(1.0) + dtype(eps / 2) != dtype(1.0):
        eps = dtype(eps / 2)
    return float(eps)

# (a)
print('Machine epsilon (iterative method):')
for dtype in [np.float16, np.float32, np.float64]:
    eps_iter = compute_epsilon(dtype)
    eps_lib = np.finfo(dtype).eps
    print(f'  {dtype.__name__:<10} computed: {eps_iter:.2e}  library: {eps_lib:.2e}  match: {np.isclose(eps_iter, eps_lib)}')

# (b) BF16 = 7 mantissa bits → ε = 2^(-7)
bf16_eps = 2**(-7)  # = 0.0078125
print(f'\nBF16: ε = 2^(-7) = {bf16_eps}')

# (c) gradient=0.001, weight=1.0 in BF16
print(f'\n(c) gradient=0.001 vs BF16 ε={bf16_eps}:')
print(f'    0.001 < {bf16_eps/2} (= ε/2)? {0.001 < bf16_eps/2}')
print(f'    → YES, 0.001 < ε/2 = {bf16_eps/2:.6f}')
print(f'    → The gradient is BELOW the rounding threshold → weight WON\'T change!')
print(f'    → This is why FP32 master weights are mandatory.')

# (d) Minimum gradient
min_grad = bf16_eps / 2  # values below this round to zero when added to 1.0
print(f'\n(d) Minimum gradient to update BF16 weight of 1.0:')
print(f'    Must be > ε/2 = {min_grad:.6f}')
print(f'    ≈ 0.39% of the weight value')
print(f'    → Any gradient magnitude below this is silently lost in BF16!')

---

## Exercise 5: BF16 vs FP16 — Range & Overflow ⭐

You switch from FP32 to FP16 and training immediately crashes with NaN.

**Task**:
- (a) What is the max representable value in FP16? In BF16? In FP32?
- (b) A gradient value of 70000.0 — can FP16 represent it? Can BF16?
- (c) Compute `exp(12.0)` in FP16 — does it overflow? What about BF16?
- (d) Why is BF16 preferred for training despite having LESS precision than FP16?

In [ ]:
# Exercise 5: Your Solution

print('Scaffold placeholder: replace the code below with your implementation and rerun this cell.')
print('The original scaffold is preserved below inside an if False block.')

if False:
    
    # (a) Max representable values
    fp16_max = None  # YOUR CODE HERE (use np.finfo)
    fp32_max = None  # YOUR CODE HERE
    bf16_max = None  # YOUR CODE HERE (compute manually: same exponent as FP32)
    
    print(f'FP16 max: {fp16_max}')
    print(f'BF16 max: {bf16_max}')
    print(f'FP32 max: {fp32_max}')
    
    # (b) Can 70000.0 be represented?
    # YOUR CODE HERE
    
    # (c) exp(12.0) in FP16 vs BF16
    # YOUR CODE HERE
    
    # (d) Why BF16 > FP16 for training? (text answer)
    # YOUR ANSWER HERE
    pass


In [ ]:
# Exercise 5: Solution ✅

# (a)
fp16_max = np.finfo(np.float16).max  # 65504
fp32_max = np.finfo(np.float32).max  # 3.4028235e+38
bf16_max = 3.3895314e+38  # same 8-bit exponent as FP32

print(f'(a) Maximum representable values:')
print(f'    FP16: {fp16_max:>14}  (5 exp bits)')
print(f'    BF16: {bf16_max:>14.4e}  (8 exp bits, same as FP32!)')
print(f'    FP32: {fp32_max:>14.4e}  (8 exp bits)')

# (b)
print(f'\n(b) Can 70000.0 be represented?')
print(f'    FP16: 70000 > {fp16_max} → OVERFLOW → inf → NaN propagates!')
print(f'    BF16: 70000 < {bf16_max:.2e} → ✓ representable')
if HAS_TORCH:
    x = torch.tensor(70000.0)
    print(f'    PyTorch verify: FP16={x.half().item()}, BF16={x.bfloat16().item()}')

# (c)
print(f'\n(c) exp(12.0):')
exp12 = np.exp(12.0)  # ≈ 162754.79
print(f'    True value: {exp12:.2f}')
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    fp16_result = np.exp(np.float16(12.0))
print(f'    FP16: {fp16_result} → {"OVERFLOW!" if np.isinf(fp16_result) else "OK"}')
print(f'    BF16: would be ~{exp12:.0f} → within range')

# (d)
print(f'\n(d) Why BF16 > FP16 for training:')
print(f'    BF16 has 8 exponent bits → same RANGE as FP32 (up to 3.4×10³⁸)')
print(f'    FP16 has 5 exponent bits → max 65504 → overflows on large gradients')
print(f'    Gradient magnitudes vary hugely during training → range matters more')
print(f'    FP16 requires loss scaling to avoid overflow; BF16 does not')

---

## Exercise 6: Floating-Point Associativity Failure ⭐

Your distributed training produces different losses on different GPU counts.
This is because floating-point addition is NOT associative.

**Task**:
- (a) Prove `(a + b) + c ≠ a + (b + c)` with `a = 1e8`, `b = 1.0`, `c = -1e8` in FP32
- (b) Explain step-by-step WHY the results differ
- (c) Show that summation ORDER affects the result for a large array of FP32 values

In [ ]:
# Exercise 6: Your Solution

print('Scaffold placeholder: replace the code below with your implementation and rerun this cell.')
print('The original scaffold is preserved below inside an if False block.')

if False:
    
    # (a) Prove non-associativity
    a = np.float32(1e8)
    b = np.float32(1.0)
    c = np.float32(-1e8)
    
    left = None   # YOUR CODE HERE: (a + b) + c
    right = None  # YOUR CODE HERE: a + (b + c)
    
    print(f'(a + b) + c = {left}')
    print(f'a + (b + c) = {right}')
    print(f'Equal? {left == right}')
    
    # (b) Step-by-step explanation
    # YOUR EXPLANATION HERE
    
    # (c) Summation order affects result
    np.random.seed(42)
    vals = np.random.randn(100000).astype(np.float32)
    # YOUR CODE HERE: compare forward vs reverse sum
    pass


In [ ]:
# Exercise 6: Solution ✅

a = np.float32(1e8)
b = np.float32(1.0)
c = np.float32(-1e8)

# (a)
left = np.float32(np.float32(a + b) + c)
right = np.float32(a + np.float32(b + c))
print(f'(a) Associativity test:')
print(f'    (a + b) + c = {left}')
print(f'    a + (b + c) = {right}')
print(f'    Equal? {left == right}  → ASSOCIATIVITY FAILS!')

# (b)
print(f'\n(b) Step-by-step:')
ab = np.float32(a + b)
print(f'    a + b = {a:.0f} + {b:.0f} = {ab:.0f}')
print(f'    → b is absorbed! 1.0 is below FP32 epsilon relative to 1e8')
print(f'    → Then {ab:.0f} + {c:.0f} = {left:.0f}')
bc = np.float32(b + c)
print(f'    b + c = {b:.0f} + ({c:.0f}) = {bc:.0f}')
print(f'    → Then {a:.0f} + {bc:.0f} = {right:.0f}  ← correct!')

# (c)
np.random.seed(42)
vals = np.random.randn(100000).astype(np.float32)
fwd = float(np.float32(0))
for v in vals: fwd = float(np.float32(fwd + v))
rev = float(np.float32(0))
for v in reversed(vals): rev = float(np.float32(rev + v))
np_sum = float(np.sum(vals))

print(f'\n(c) Summation order matters:')
print(f'    Forward sum:  {fwd:.6f}')
print(f'    Reverse sum:  {rev:.6f}')
print(f'    NumPy sum:    {np_sum:.6f}')
print(f'    Difference:   {abs(fwd - rev):.6f}')
print(f'    → Different GPU reduction trees produce different sums!')
print(f'    → This is why distributed training is not bitwise reproducible')

---

## Exercise 7: INT8 Symmetric Quantization — Full Pipeline ⭐⭐

You're deploying a model to INT8. Implement the complete quantize → compute → dequantize pipeline.

**Task**: Given weight tensor $W = [-1.2, 0.3, -0.7, 0.9, 0.0, -0.1, 0.5, -0.4]$:
- (a) Compute the scale factor $s = \max(|W|) / 127$
- (b) Quantize all values: $W_q = \text{clamp}(\text{round}(W / s), -128, 127)$
- (c) Dequantize: $\hat{W} = W_q \times s$
- (d) Compute maximum absolute error and SQNR in dB

This is Exercise 2 from notes.md, fully implemented.

In [ ]:
# Exercise 7: Your Solution

print('Scaffold placeholder: replace the code below with your implementation and rerun this cell.')
print('The original scaffold is preserved below inside an if False block.')

if False:
    
    W = np.array([-1.2, 0.3, -0.7, 0.9, 0.0, -0.1, 0.5, -0.4], dtype=np.float32)
    
    # (a) Scale factor
    s = None  # YOUR CODE HERE
    print(f'Scale s = {s}')
    
    # (b) Quantize
    W_q = None  # YOUR CODE HERE
    print(f'W_q = {W_q}')
    
    # (c) Dequantize
    W_hat = None  # YOUR CODE HERE
    print(f'W_hat = {W_hat}')
    
    # (d) Error analysis
    max_error = None  # YOUR CODE HERE
    sqnr_db = None    # YOUR CODE HERE: 10 * log10(signal_power / noise_power)
    print(f'Max |error| = {max_error}')
    print(f'SQNR = {sqnr_db:.1f} dB')
    pass


In [ ]:
# Exercise 7: Solution ✅

W = np.array([-1.2, 0.3, -0.7, 0.9, 0.0, -0.1, 0.5, -0.4], dtype=np.float32)

# (a)
s = np.max(np.abs(W)) / 127
print(f'(a) Scale s = max(|W|) / 127 = {np.max(np.abs(W)):.1f} / 127 = {s:.8f}')

# (b)
W_q = np.clip(np.round(W / s), -128, 127).astype(np.int8)
print(f'\n(b) Quantized values:')
print(f'    {"W":>7} {"W/s":>8} {"round":>6} {"W_q":>5}')
    
for w, q in zip(W, W_q):
    print(f'    {w:>7.2f} {w/s:>8.2f} {np.round(w/s):>6.0f} {int(q):>5}')

# (c)
W_hat = W_q.astype(np.float32) * s
print(f'\n(c) Dequantized:')
for w, wh, e in zip(W, W_hat, np.abs(W - W_hat)):
    print(f'    {w:>7.4f} → {wh:>7.4f}  error={e:.6f}')

# (d)
errors = np.abs(W - W_hat)
max_error = np.max(errors)
signal_power = np.mean(W**2)
noise_power = np.mean((W - W_hat)**2)
sqnr_db = 10 * np.log10(signal_power / noise_power)

print(f'\n(d) Error analysis:')
print(f'    Max |error| = {max_error:.6f}  (theoretical max = s/2 = {s/2:.6f})')
print(f'    Signal power = {signal_power:.6f}')
print(f'    Noise power  = {noise_power:.8f}')
print(f'    SQNR = {sqnr_db:.1f} dB')
print(f'    Theory: 8-bit → ~{6.02*8:.1f} dB (6 dB per bit rule)')
print(f'    Memory: {W.nbytes} bytes → {W_q.nbytes} bytes ({W.nbytes/W_q.nbytes:.0f}× compression)')

---

## Exercise 8: Per-Channel vs Per-Tensor Quantization ⭐⭐

Your INT8 model loses significant quality. The weight matrix has channels with very different magnitudes.

**Task**: Implement both `quantize_per_tensor(W)` and `quantize_per_channel(W)` for INT8.
- Compare MSE for a matrix where channel 0 has 100× scale of channel 1
- Show which channel suffers most under per-tensor quantization
- Explain WHY per-channel is better

In [ ]:
# Exercise 8: Your Solution

print('Scaffold placeholder: replace the code below with your implementation and rerun this cell.')
print('The original scaffold is preserved below inside an if False block.')

if False:
    
    def quantize_per_tensor(W: np.ndarray) -> Tuple[np.ndarray, float, np.ndarray]:
        """Per-tensor INT8: single scale for entire matrix.
        Returns: (quantized, scale, dequantized)"""
        # YOUR CODE HERE
        pass
    
    def quantize_per_channel(W: np.ndarray) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
        """Per-channel INT8: one scale per output channel (row).
        Returns: (quantized, scales, dequantized)"""
        # YOUR CODE HERE
        pass
    
    # Test with outlier channels
    np.random.seed(42)
    W = np.random.randn(4, 256).astype(np.float32)
    W[0] *= 10     # Large channel
    W[1] *= 0.01   # Tiny channel
    
    # Compare per-channel MSE for each method
    pass


In [ ]:
# Exercise 8: Solution ✅

def quantize_per_tensor(W):
    s = np.max(np.abs(W)) / 127
    q = np.clip(np.round(W / s), -128, 127).astype(np.int8)
    deq = q.astype(np.float32) * s
    return q, s, deq

def quantize_per_channel(W):
    n_ch = W.shape[0]
    scales = np.zeros(n_ch, dtype=np.float32)
    q = np.zeros_like(W, dtype=np.int8)
    for c in range(n_ch):
        scales[c] = np.max(np.abs(W[c])) / 127
        if scales[c] > 0:
            q[c] = np.clip(np.round(W[c] / scales[c]), -128, 127).astype(np.int8)
    deq = q.astype(np.float32) * scales[:, None]
    return q, scales, deq

np.random.seed(42)
W = np.random.randn(4, 256).astype(np.float32)
W[0] *= 10; W[1] *= 0.01

_, _, deq_t = quantize_per_tensor(W)
_, scales_c, deq_c = quantize_per_channel(W)

print(f'{"Channel":>4} {"σ(W)":>8} {"Scale":>10} {"Per-Tensor MSE":>16} {"Per-Channel MSE":>16} {"Improvement":>12}')
print('-' * 72)
for c in range(4):
    mse_t = np.mean((W[c] - deq_t[c])**2)
    mse_c = np.mean((W[c] - deq_c[c])**2)
    imp = mse_t / mse_c if mse_c > 0 else float('inf')
    flag = ' ← outlier' if c in [0, 1] else ''
    print(f'{c:>4} {np.std(W[c]):>8.4f} {scales_c[c]:>10.6f} {mse_t:>16.8f} {mse_c:>16.8f} {imp:>10.1f}×{flag}')

total_t = np.mean((W - deq_t)**2)
total_c = np.mean((W - deq_c)**2)
print(f'\nTotal MSE: per-tensor={total_t:.8f}, per-channel={total_c:.8f} ({total_t/total_c:.1f}× better)')
print(f'\n→ Channel 1 (tiny weights) gets crushed: its values are far below the global scale')
print(f'→ Per-channel gives each channel its own scale → utilizes full INT8 range')

---

## Exercise 9: INT4 Group Quantization — W4A16 ⭐⭐

INT4 has only 16 levels [-8, 7]. Per-tensor would be unusable. Group quantization saves it.

**Task**: Implement `quantize_int4_grouped(W, group_size=64)` that:
- Splits each row into groups of `group_size` elements
- Applies symmetric INT4 quantization per group
- Returns quantized values, scales, and dequantized reconstruction
- Compare quality with group_size = 32, 64, 128, 256

In [ ]:
# Exercise 9: Your Solution

print('Scaffold placeholder: replace the code below with your implementation and rerun this cell.')
print('The original scaffold is preserved below inside an if False block.')

if False:
    
    def quantize_int4_grouped(W: np.ndarray, group_size: int = 64):
        """INT4 group quantization. Range: [-8, 7].
        Returns: (quantized, scales_per_group, dequantized)"""
        # YOUR CODE HERE
        # For each row, split into groups of `group_size`
        # For each group: s = max(|group|) / 7
        #                 q = clamp(round(group/s), -8, 7)
        pass
    
    # Test
    np.random.seed(42)
    W = np.random.randn(64, 256).astype(np.float32) * 0.02
    
    for gs in [32, 64, 128, 256]:
        _, _, deq = quantize_int4_grouped(W, group_size=gs)
        mse = np.mean((W - deq)**2)
        n_scales = (W.shape[0] * W.shape[1]) // gs
        print(f'group_size={gs:>3}: MSE={mse:.10f}, #scales={n_scales}')
    pass


In [ ]:
# Exercise 9: Solution ✅

def quantize_int4_grouped(W, group_size=64):
    rows, cols = W.shape
    q = np.zeros_like(W, dtype=np.int8)
    deq = np.zeros_like(W)
    all_scales = []

    for r in range(rows):
        for start in range(0, cols, group_size):
            end = min(start + group_size, cols)
            group = W[r, start:end]
            alpha = np.max(np.abs(group))
            s = alpha / 7.0 if alpha > 0 else 1.0
            q[r, start:end] = np.clip(np.round(group / s), -8, 7).astype(np.int8)
            deq[r, start:end] = q[r, start:end].astype(np.float32) * s
            all_scales.append(s)

    return q, np.array(all_scales), deq

np.random.seed(42)
W = np.random.randn(64, 256).astype(np.float32) * 0.02

print(f'{"Group Size":>10} {"MSE":>14} {"# Scales":>10} {"Scale overhead":>16}')
print('-' * 55)
for gs in [32, 64, 128, 256]:
    _, scales, deq = quantize_int4_grouped(W, group_size=gs)
    mse = np.mean((W - deq)**2)
    n_scales = len(scales)
    overhead_bytes = n_scales * 2  # FP16 scales
    weight_bytes = W.size // 2     # 4-bit = 0.5 bytes each
    overhead_pct = overhead_bytes / weight_bytes * 100
    print(f'{gs:>10} {mse:>14.10f} {n_scales:>10} {overhead_pct:>14.1f}%')

print(f'\n→ Smaller groups → better quality but more scale overhead')
print(f'→ group_size=128 is the standard choice (GPTQ, AWQ)')
print(f'→ group_size=64 in QLoRA for 4-bit quantization')
print(f'→ Only 16 quantization levels but quality is surprisingly good with grouping!')

---

## Exercise 10: NF4 — Optimal Quantization for Gaussian Weights ⭐⭐

QLoRA uses NF4 (Normal Float 4-bit): quantization levels are optimized for N(0,1) distribution.

**Task**:
- (a) Compute the 16 NF4 levels as quantiles of N(0,1) at evenly-spaced probabilities
- (b) Quantize 10000 normally-distributed weights using NF4
- (c) Compare MSE with uniform INT4 quantization
- (d) Why is NF4 better for neural network weights?

In [ ]:
# Exercise 10: Your Solution

print('Scaffold placeholder: replace the code below with your implementation and rerun this cell.')
print('The original scaffold is preserved below inside an if False block.')

if False:
    
    from scipy.stats import norm
    
    def compute_nf4_levels() -> np.ndarray:
        """Compute NF4 quantization levels as quantiles of N(0,1).
        16 levels at probabilities: 1/32, 3/32, 5/32, ..., 31/32.
        Normalize to [-1, 1]."""
        # YOUR CODE HERE
        pass
    
    def quantize_nf4(x: np.ndarray, levels: np.ndarray) -> np.ndarray:
        """Map each value to nearest NF4 level."""
        # YOUR CODE HERE
        # Normalize x to [-1, 1], then find closest level for each value
        pass
    
    # (a) Compute levels
    nf4 = compute_nf4_levels()
    print(f'NF4 levels: {nf4}')
    
    # (b)-(c) Compare NF4 vs INT4
    np.random.seed(42)
    weights = np.random.randn(10000).astype(np.float32)
    pass


In [ ]:
# Exercise 10: Solution ✅

from scipy.stats import norm

def compute_nf4_levels() -> np.ndarray:
    n = 16
    probs = np.linspace(1/(2*n), 1 - 1/(2*n), n)
    levels = norm.ppf(probs)
    return levels / np.max(np.abs(levels))  # normalize to [-1, 1]

def quantize_nf4(x, levels):
    scale = np.max(np.abs(x))
    x_norm = x / scale
    indices = np.array([np.argmin(np.abs(levels - v)) for v in x_norm])
    return levels[indices] * scale

# (a)
nf4 = compute_nf4_levels()
print(f'(a) NF4 levels (16 quantiles of N(0,1), normalized):')
for i, l in enumerate(nf4):
    bar = '█' * int((l + 1) * 20)
    print(f'    {i:>2}: {l:>7.4f}  {bar}')

# (b)-(c)
np.random.seed(42)
weights = np.random.randn(10000).astype(np.float32)

# NF4
recon_nf4 = quantize_nf4(weights, nf4)
mse_nf4 = np.mean((weights - recon_nf4)**2)

# Uniform INT4
alpha = np.max(np.abs(weights))
s = alpha / 7
q_int4 = np.clip(np.round(weights / s), -8, 7)
recon_int4 = q_int4 * s
mse_int4 = np.mean((weights - recon_int4)**2)

print(f'\n(b)-(c) Quantization quality on 10K N(0,1) weights:')
print(f'    NF4 MSE:  {mse_nf4:.6f}')
print(f'    INT4 MSE: {mse_int4:.6f}')
print(f'    NF4 is {mse_int4/mse_nf4:.1f}× better!')

print(f'\n(d) Why NF4 is better:')
print(f'    Neural network weights are approximately Gaussian (bell-shaped)')
print(f'    NF4 places more levels near zero where data density is highest')
print(f'    INT4 wastes levels on sparse tails (values > 2σ are rare but get many levels)')
print(f'    → NF4 is the optimal 4-bit format for Gaussian data (used in QLoRA)')

---

## Exercise 11: Catastrophic Cancellation in Variance ⭐⭐

LayerNorm computes variance. The naive formula `var = E[x²] - E[x]²` is unstable.

**Task**:
- (a) Compute variance of data with mean=10⁶ and std=0.01 using the naive formula in FP32
- (b) Compute using the stable formula `var = E[(x - μ)²]`
- (c) Compare with FP64 ground truth
- (d) Explain why RMSNorm avoids this problem

In [ ]:
# Exercise 11: Your Solution

print('Scaffold placeholder: replace the code below with your implementation and rerun this cell.')
print('The original scaffold is preserved below inside an if False block.')

if False:
    
    np.random.seed(42)
    data = (np.float32(1e6) + np.random.randn(10000).astype(np.float32) * 0.01)
    
    # True variance in FP64
    true_var = np.var(data.astype(np.float64))
    
    # (a) Naive formula: var = E[x²] - E[x]²
    naive_var = None  # YOUR CODE HERE
    
    # (b) Stable formula: var = E[(x - mean)²]
    stable_var = None  # YOUR CODE HERE
    
    print(f'True variance (FP64):  {true_var:.10f}')
    print(f'Naive (FP32):          {naive_var}')
    print(f'Stable (FP32):         {stable_var}')
    pass


In [ ]:
# Exercise 11: Solution ✅

np.random.seed(42)
data = (np.float32(1e6) + np.random.randn(10000).astype(np.float32) * 0.01)
true_var = float(np.var(data.astype(np.float64)))

# (a) Naive
mean_sq = np.float32(np.mean(data.astype(np.float32)**2))
sq_mean = np.float32(np.mean(data.astype(np.float32)))**2
naive_var = float(np.float32(mean_sq - sq_mean))

# (b) Stable
centered = data - np.float32(np.mean(data))
stable_var = float(np.float32(np.mean(centered**2)))

print(f'(a) Naive:  var = E[x²] - E[x]²')
print(f'    E[x²]  = {mean_sq:.2f}')
print(f'    E[x]²  = {sq_mean:.2f}')
print(f'    Difference = {naive_var}  ← GARBAGE (should be ~{true_var:.6f})')
print(f'    Relative error: {abs(naive_var - true_var) / true_var:.0%}')

print(f'\n(b) Stable: var = E[(x - μ)²]')
print(f'    Result = {stable_var:.10f}')
print(f'    Relative error: {abs(stable_var - true_var) / true_var:.1%}')

print(f'\n(c) Ground truth (FP64): {true_var:.10f}')

print(f'\n(d) Why RMSNorm avoids this:')
print(f'    LayerNorm: (x - μ) / √var → subtraction of nearly-equal values → cancellation')
print(f'    RMSNorm:  x / √(mean(x²)) → no subtraction, only x² (always positive) → no cancellation')
print(f'    → RMSNorm is used in LLaMA, Mistral, Gemma (all 2024+ architectures)')

---

## Exercise 12: Kahan Compensated Summation ⭐⭐

You're accumulating gradients across 1M micro-batches in FP32. Naive summation drifts.

**Task**: Implement `kahan_sum(values)` that:
- Tracks rounding error with a compensation variable
- Works in FP32 (no FP64 upcast)
- Compare with naive sum and numpy sum on 1M values of 1e-5

```
Algorithm:
  comp = 0  (compensation for lost low-order bits)
  for each value:
      y = value - comp
      temp = total + y
      comp = (temp - total) - y   ← captures what was lost
      total = temp
```

In [ ]:
# Exercise 12: Your Solution

print('Scaffold placeholder: replace the code below with your implementation and rerun this cell.')
print('The original scaffold is preserved below inside an if False block.')

if False:
    
    def kahan_sum(values: np.ndarray) -> float:
        """Kahan compensated summation in FP32."""
        # YOUR CODE HERE
        pass
    
    # Test: sum 1,000,000 × 1e-5 (expected: 10.0)
    vals = np.full(1_000_000, 1e-5, dtype=np.float32)
    expected = 10.0
    
    result = kahan_sum(vals)
    print(f'Kahan sum: {result:.8f}  (expected: {expected})')
    pass


In [ ]:
# Exercise 12: Solution ✅

def kahan_sum(values):
    total = np.float32(0.0)
    comp = np.float32(0.0)
    for v in values:
        v = np.float32(v)
        y = np.float32(v - comp)
        temp = np.float32(total + y)
        comp = np.float32(np.float32(temp - total) - y)
        total = temp
    return float(total)

# Compare all methods
vals = np.full(1_000_000, 1e-5, dtype=np.float32)
expected = 10.0

# Naive
naive = np.float32(0.0)
for v in vals[:100000]:  # first 100K for speed
    naive = np.float32(naive + v)
naive = float(naive) * 10  # scale up

# Kahan (on subset for speed)
kahan = kahan_sum(vals[:100000]) * 10

# NumPy (pairwise)
np_result = float(np.sum(vals))

print(f'{"Method":<20} {"Result":>12} {"Error":>12} {"Relative":>12}')
print('-' * 60)
for name, result in [('Naive FP32', naive), ('Kahan', kahan), ('NumPy (pairwise)', np_result)]:
    err = abs(result - expected)
    rel = err / expected
    print(f'{name:<20} {result:>12.6f} {err:>12.6f} {rel:>12.2e}')

print(f'\n→ Kahan summation tracks what was "lost" at each step')
print(f'→ Essential for gradient accumulation across many micro-batches')
print(f'→ DeepSpeed and Megatron-LM use compensated summation internally')

---

## Exercise 13: Numerically Stable Softmax ⭐⭐

Attention logits in your transformer reach 1000+. Naive `exp(x)` overflows.

**Task**:
- (a) Apply naive softmax to `[1000, 1001, 1002]` — show it produces NaN
- (b) Implement `stable_softmax(z)` using max subtraction trick
- (c) Prove mathematically that `softmax(z) = softmax(z - max(z))`
- (d) Handle batched input (2D array, softmax along last axis)

In [ ]:
# Exercise 13: Your Solution

print('Scaffold placeholder: replace the code below with your implementation and rerun this cell.')
print('The original scaffold is preserved below inside an if False block.')

if False:
    
    def naive_softmax(z: np.ndarray) -> np.ndarray:
        """Naive softmax — will overflow."""
        # YOUR CODE HERE
        pass
    
    def stable_softmax(z: np.ndarray) -> np.ndarray:
        """Numerically stable softmax. Works for 1D and 2D."""
        # YOUR CODE HERE
        pass
    
    # Tests
    z_normal = np.array([1.0, 2.0, 3.0])
    z_large = np.array([1000.0, 1001.0, 1002.0])
    z_batch = np.array([[1, 2, 3], [1000, 1001, 1002]], dtype=np.float64)
    
    print(f'Normal: {stable_softmax(z_normal)}')
    print(f'Large:  {stable_softmax(z_large)}')
    print(f'Batch:\n{stable_softmax(z_batch)}')
    pass


In [ ]:
# Exercise 13: Solution ✅

def naive_softmax(z):
    exp_z = np.exp(z)
    return exp_z / np.sum(exp_z, axis=-1, keepdims=True)

def stable_softmax(z):
    z = np.asarray(z, dtype=np.float64)
    z_shifted = z - np.max(z, axis=-1, keepdims=True)
    exp_z = np.exp(z_shifted)
    return exp_z / np.sum(exp_z, axis=-1, keepdims=True)

# (a) Naive breaks on large values
z_large = np.array([1000.0, 1001.0, 1002.0])
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    naive_result = naive_softmax(z_large)
print(f'(a) Naive softmax([1000,1001,1002]): {naive_result} ← inf/inf = NaN!')

# (b) Stable works
stable_result = stable_softmax(z_large)
print(f'(b) Stable softmax([1000,1001,1002]): {stable_result}')
assert not np.any(np.isnan(stable_result))
assert np.isclose(stable_result.sum(), 1.0)

# (c) Mathematical proof
print(f'\n(c) Proof that softmax(z) = softmax(z - c) for any constant c:')
print(f'    softmax(z-c)_i = exp(z_i - c) / Σ exp(z_j - c)')
print(f'                   = [exp(z_i) · exp(-c)] / [exp(-c) · Σ exp(z_j)]')
print(f'                   = exp(z_i) / Σ exp(z_j)')
print(f'                   = softmax(z)_i')
print(f'    Choosing c = max(z) ensures max exponent is e^0 = 1 → no overflow. ✓')

# (d) Batched
z_batch = np.array([[1, 2, 3], [1000, 1001, 1002]], dtype=np.float64)
result = stable_softmax(z_batch)
print(f'\n(d) Batch softmax:')
print(f'    Row 0: {result[0]} (sum={result[0].sum():.6f})')
print(f'    Row 1: {result[1]} (sum={result[1].sum():.6f})')
print(f'\n→ This is used in EVERY attention computation in EVERY transformer ✓')

---

## Exercise 14: Stable Cross-Entropy via Log-Sum-Exp ⭐⭐

Cross-entropy loss: $L = -\log(\text{softmax}(z)_{\text{target}})$

Computing softmax then log is unstable. The identity `log_softmax(z)_i = z_i - LSE(z)` avoids both.

**Task**:
- (a) Implement `stable_logsumexp(z)` = $\max(z) + \log\sum \exp(z - \max(z))$
- (b) Implement `stable_cross_entropy(logits, target_index)`
- (c) Test with logits = [1000, 1001, 1002], target = class 2
- (d) Verify against PyTorch's `nn.CrossEntropyLoss`

In [ ]:
# Exercise 14: Your Solution

print('Scaffold placeholder: replace the code below with your implementation and rerun this cell.')
print('The original scaffold is preserved below inside an if False block.')

if False:
    
    def stable_logsumexp(z: np.ndarray) -> float:
        """log(sum(exp(z))) computed stably."""
        # YOUR CODE HERE
        pass
    
    def stable_cross_entropy(logits: np.ndarray, target: int) -> float:
        """Cross-entropy: -log_softmax(logits)[target]
        = -(logits[target] - logsumexp(logits))"""
        # YOUR CODE HERE
        pass
    
    # Test
    logits = np.array([1000.0, 1001.0, 1002.0])
    loss = stable_cross_entropy(logits, target=2)
    print(f'Loss: {loss:.4f}')
    pass


In [ ]:
# Exercise 14: Solution ✅

def stable_logsumexp(z):
    z = np.asarray(z, dtype=np.float64)
    m = np.max(z)
    return float(m + np.log(np.sum(np.exp(z - m))))

def stable_cross_entropy(logits, target):
    logits = np.asarray(logits, dtype=np.float64)
    lse = stable_logsumexp(logits)
    return float(lse - logits[target])

# Test with extreme logits
logits = np.array([1000.0, 1001.0, 1002.0])
loss = stable_cross_entropy(logits, target=2)
print(f'logits = {logits}')
print(f'LSE = {stable_logsumexp(logits):.6f}')
print(f'Loss = LSE - logits[2] = {stable_logsumexp(logits):.4f} - {logits[2]:.1f} = {loss:.4f}')
assert not np.isnan(loss)

# Verify: log_softmax = logits - LSE should give valid log-probs
lse = stable_logsumexp(logits)
log_probs = logits - lse
probs = np.exp(log_probs)
print(f'\nLog-probs: {log_probs}')
print(f'Probs:     {probs}  (sum = {probs.sum():.6f})')

# Compare with PyTorch
if HAS_TORCH:
    pt_loss = nn.CrossEntropyLoss()(torch.tensor([logits]), torch.tensor([2])).item()
    print(f'\nPyTorch CE: {pt_loss:.4f}  Match: {np.isclose(loss, pt_loss, atol=0.01)} ✓')

print(f'\n→ NEVER compute log(softmax(z)) — compute z - logsumexp(z) instead')
print(f'→ This is why PyTorch has F.log_softmax() and F.cross_entropy()')

---

## Exercise 15: SQNR — The 6 dB/bit Rule ⭐⭐⭐

When choosing quantization bit width, SQNR tells you how much noise you're adding.

**Task**:
- (a) Implement `compute_sqnr(x, x_hat)` that returns SQNR in dB
- (b) Compute SQNR for 1–8 bit quantization of a uniform signal
- (c) Verify the 6.02 dB/bit rule: SQNR ≈ 6.02 × bits
- (d) Show that Gaussian signals have slightly worse SQNR (why?)

In [ ]:
# Exercise 15: Your Solution

print('Scaffold placeholder: replace the code below with your implementation and rerun this cell.')
print('The original scaffold is preserved below inside an if False block.')

if False:
    
    def compute_sqnr(x: np.ndarray, x_hat: np.ndarray) -> float:
        """SQNR in dB = 10 * log10(signal_power / noise_power)"""
        # YOUR CODE HERE
        pass
    
    def quantize_symmetric(x: np.ndarray, bits: int) -> np.ndarray:
        """Symmetric quantization to given bit width."""
        # YOUR CODE HERE
        pass
    
    # Test on uniform and normal signals
    np.random.seed(42)
    x_uniform = np.random.uniform(-1, 1, 100000).astype(np.float32)
    x_normal = np.random.randn(100000).astype(np.float32)
    
    for bits in range(1, 9):
        x_hat_u = quantize_symmetric(x_uniform, bits)
        x_hat_n = quantize_symmetric(x_normal, bits)
        theory = 6.02 * bits
        print(f'{bits}-bit: theory={theory:.1f} dB  uniform={compute_sqnr(x_uniform, x_hat_u):.1f}  '
              f'normal={compute_sqnr(x_normal, x_hat_n):.1f}')
    pass


In [ ]:
# Exercise 15: Solution ✅

def compute_sqnr(x, x_hat):
    signal_power = np.mean(x**2)
    noise_power = np.mean((x - x_hat)**2)
    if noise_power == 0:
        return float('inf')
    return 10 * np.log10(signal_power / noise_power)

def quantize_symmetric(x, bits):
    alpha = np.max(np.abs(x))
    qmax = 2**(bits-1) - 1
    s = alpha / qmax if alpha > 0 else 1.0
    q = np.clip(np.round(x / s), -qmax-1, qmax)
    return q * s

np.random.seed(42)
x_u = np.random.uniform(-1, 1, 100000).astype(np.float32)
x_n = np.random.randn(100000).astype(np.float32)

print(f'{"Bits":>5} {"Theory":>8} {"Uniform":>9} {"Normal":>9} {"Gap (dB)":>10}')
print('-' * 45)
for bits in range(1, 9):
    theory = 6.02 * bits
    sqnr_u = compute_sqnr(x_u, quantize_symmetric(x_u, bits))
    sqnr_n = compute_sqnr(x_n, quantize_symmetric(x_n, bits))
    print(f'{bits:>5} {theory:>8.1f} {sqnr_u:>9.1f} {sqnr_n:>9.1f} {sqnr_u - sqnr_n:>10.1f}')

print(f'\n→ Each additional bit adds ~6 dB (= 4× less noise power)')
print(f'→ Normal signal has lower SQNR because outliers waste quantization levels')
print(f'→ This is precisely why NF4 (non-uniform) outperforms INT4 (uniform) for Gaussian data')
print(f'→ At 4-bit: ~24 dB, noise power is ~0.4% of signal → acceptable for inference')
print(f'→ At 8-bit: ~48 dB, noise power is ~0.001% of signal → nearly lossless')

---

## Exercise 16: Lloyd-Max Optimal Quantization ⭐⭐⭐

The Lloyd-Max algorithm finds optimal quantization levels for any data distribution.
NF4 is the Lloyd-Max solution for N(0,1) at 4-bit.

**Task**: Implement the Lloyd-Max algorithm:
1. Initialize levels uniformly
2. Set boundaries = midpoints between adjacent levels
3. Set levels = centroids of data in each bin
4. Repeat until convergence
5. Compare quality with uniform quantization

In [ ]:
# Exercise 16: Your Solution

print('Scaffold placeholder: replace the code below with your implementation and rerun this cell.')
print('The original scaffold is preserved below inside an if False block.')

if False:
    
    def lloyd_max(data: np.ndarray, n_levels: int, max_iter: int = 100) -> np.ndarray:
        """Lloyd-Max algorithm for optimal quantization.
        Returns: optimized levels array."""
        # Step 0: Initialize levels uniformly
        levels = np.linspace(np.min(data), np.max(data), n_levels)
    
        for _ in range(max_iter):
            # Step 1: boundaries = midpoints between levels
            # YOUR CODE HERE
    
            # Step 2: levels = centroids of data in each bin
            # YOUR CODE HERE
    
            # Step 3: check convergence
            # YOUR CODE HERE
            pass
    
        return levels
    
    # Test
    np.random.seed(42)
    data = np.random.randn(100000)
    levels = lloyd_max(data, 16)
    print(f'Optimal 4-bit levels for N(0,1):')
    print(levels.round(3))
    pass


In [ ]:
# Exercise 16: Solution ✅

def lloyd_max(data, n_levels, max_iter=100):
    levels = np.linspace(np.min(data), np.max(data), n_levels)

    for iteration in range(max_iter):
        # Boundaries = midpoints between levels
        boundaries = (levels[:-1] + levels[1:]) / 2

        # Levels = centroids of data in each bin
        new_levels = np.zeros_like(levels)
        all_bounds = np.concatenate([[-np.inf], boundaries, [np.inf]])
        for i in range(n_levels):
            mask = (data >= all_bounds[i]) & (data < all_bounds[i+1])
            if np.sum(mask) > 0:
                new_levels[i] = np.mean(data[mask])
            else:
                new_levels[i] = levels[i]

        if np.allclose(levels, new_levels, atol=1e-8):
            print(f'    Converged in {iteration+1} iterations')
            break
        levels = new_levels

    return levels

np.random.seed(42)
data = np.random.randn(100000).astype(np.float64)

print('Lloyd-Max Optimal Quantization for N(0,1):')
for bits in [2, 3, 4]:
    n = 2**bits
    print(f'\n  {bits}-bit ({n} levels):')
    levels = lloyd_max(data, n)
    print(f'    Levels: {np.array2string(levels, precision=3, separator=", ")}')

    # Compare with uniform
    # Lloyd-Max quantize
    boundaries = (levels[:-1] + levels[1:]) / 2
    indices = np.digitize(data, boundaries)
    lm_recon = levels[indices]
    lm_mse = np.mean((data - lm_recon)**2)

    # Uniform quantize
    u_recon = quantize_symmetric(data.astype(np.float32), bits)
    u_mse = np.mean((data - u_recon)**2)

    print(f'    Lloyd-Max MSE: {lm_mse:.6f}')
    print(f'    Uniform MSE:   {u_mse:.6f}')
    print(f'    Improvement:   {u_mse/lm_mse:.2f}×')

print(f'\n→ Lloyd-Max levels at 4-bit match the NF4 levels used in QLoRA!')
print(f'→ The algorithm is the mathematical foundation of non-uniform quantization')

---

## Exercise 17: Memory Budget Calculator ⭐⭐⭐

Before allocating GPU resources, you must know: will my training/inference fit?

**Task**: Implement `memory_budget(n_params, mode)` that calculates total GPU memory for:
- `train_fp32`: weights(FP32) + grads(FP32) + Adam(2×FP32)
- `train_bf16`: master(FP32) + working(BF16) + grads(BF16) + Adam(2×FP32)
- `qlora_nf4`: base(NF4=0.5B/param) + LoRA(~1% params in BF16) + Adam for LoRA only
- `inference_bf16`, `inference_int8`, `inference_int4`

Test on 7B, 13B, 70B, 405B parameter models.

In [ ]:
# Exercise 17: Your Solution

print('Scaffold placeholder: replace the code below with your implementation and rerun this cell.')
print('The original scaffold is preserved below inside an if False block.')

if False:
    
    def memory_budget(n_params: float, mode: str) -> dict:
        """Calculate memory budget in GB.
        Returns dict with component breakdown and total."""
        # YOUR CODE HERE
        # Key: 1 GB = 10^9 bytes
        # FP32 = 4 bytes, BF16/FP16 = 2 bytes, INT8 = 1 byte, INT4 = 0.5 bytes
        # Adam stores m (momentum) and v (variance), each FP32
        pass
    
    # Test
    for model, params in [('7B', 7e9), ('70B', 70e9)]:
        print(f'\n--- {model} ---')
        for mode in ['train_fp32', 'train_bf16', 'qlora_nf4', 'inference_int4']:
            result = memory_budget(params, mode)
            print(f'  {mode:<18}: {result["total_gb"]:.1f} GB')
    pass


In [ ]:
# Exercise 17: Solution ✅

def memory_budget(n_params, mode):
    bytes_per = {
        'train_fp32': {
            'weights': 4, 'gradients': 4, 'adam_m': 4, 'adam_v': 4},
        'train_bf16': {
            'master_weights': 4, 'working_weights': 2, 'gradients': 2,
            'adam_m': 4, 'adam_v': 4},
        'qlora_nf4': {
            'base_weights_nf4': 0.5, 'lora_weights': 0.02,
            'lora_grads': 0.02, 'adam_m': 0.02, 'adam_v': 0.02},
        'inference_bf16': {'weights': 2},
        'inference_int8': {'weights': 1},
        'inference_int4': {'weights': 0.5},
    }
    config = bytes_per[mode]
    components = {k: v * n_params / 1e9 for k, v in config.items()}
    total = sum(components.values())
    return {'components': components, 'total_gb': total}

# Full comparison table
print(f'{"Model":>6} {"FP32 Train":>12} {"BF16 Train":>12} {"QLoRA NF4":>12} '
      f'{"Infer BF16":>12} {"Infer INT8":>12} {"Infer INT4":>12}')
print('=' * 80)
for name, p in [('7B', 7e9), ('13B', 13e9), ('70B', 70e9), ('405B', 405e9)]:
    results = [memory_budget(p, m)['total_gb'] for m in 
               ['train_fp32', 'train_bf16', 'qlora_nf4',
                'inference_bf16', 'inference_int8', 'inference_int4']]
    fits = ['✓' if r <= 80 else '✗' for r in results]
    print(f'{name:>6}', '  '.join(f'{r:>7.0f} GB{f}' for r, f in zip(results, fits)))

print(f'\nKey insights:')
print(f'  → FP32 training of 70B: 1120 GB → needs 14+ A100-80GB GPUs')
print(f'  → QLoRA NF4 fine-tuning of 70B: ~36 GB → fits on single GPU!')
print(f'  → INT4 inference of 70B: 35 GB → fits on single GPU!')
print(f'  → INT4 inference of 405B: 202 GB → needs 3 GPUs')

---

## Exercise 18: KV Cache Memory Analysis ⭐⭐⭐

For long-context LLM serving, the KV cache often exceeds model weight memory.

**Task**: Implement `kv_cache_memory(n_layers, n_kv_heads, d_head, seq_len, batch, bpe)` that:
- Calculates KV cache memory: `2 × layers × kv_heads × d_head × seq_len × batch × bytes_per_element`
- Compare BF16 vs INT8 vs INT4 KV cache for LLaMA-3 70B at various context lengths
- Calculate the total inference memory (model weights + KV cache)

In [ ]:
# Exercise 18: Your Solution

print('Scaffold placeholder: replace the code below with your implementation and rerun this cell.')
print('The original scaffold is preserved below inside an if False block.')

if False:
    
    def kv_cache_gb(n_layers, n_kv_heads, d_head, seq_len, batch=1, bpe=2.0):
        """KV cache memory in GB. bpe = bytes per element."""
        # YOUR CODE HERE
        # 2 (K and V) × layers × kv_heads × d_head × seq × batch × bpe / 1e9
        pass
    
    # LLaMA-3 70B config: 80 layers, 8 KV heads (GQA), d_head=128
    for seq_len in [2048, 8192, 32768, 131072]:
        mem = kv_cache_gb(80, 8, 128, seq_len)
        print(f'seq_len={seq_len:>7,}: KV cache = {mem:.2f} GB')
    pass


In [ ]:
# Exercise 18: Solution ✅

def kv_cache_gb(n_layers, n_kv_heads, d_head, seq_len, batch=1, bpe=2.0):
    return 2 * n_layers * n_kv_heads * d_head * seq_len * batch * bpe / 1e9

# LLaMA-3 70B: 80 layers, 8 KV heads (GQA), d_head=128
print('LLaMA-3 70B KV Cache Analysis')
print('=' * 70)
print(f'{"Seq Len":>10} {"BF16 (GB)":>10} {"INT8 (GB)":>10} {"INT4 (GB)":>10} '
      f'{"Weights+KV":>12} {"Fits 80GB?":>11}')
print('-' * 60)
model_weight_int4 = 70e9 * 0.5 / 1e9  # INT4 model weights ≈ 35 GB
for seq_len in [2048, 8192, 32768, 131072]:
    bf16 = kv_cache_gb(80, 8, 128, seq_len, bpe=2)
    int8 = kv_cache_gb(80, 8, 128, seq_len, bpe=1)
    int4 = kv_cache_gb(80, 8, 128, seq_len, bpe=0.5)
    total = model_weight_int4 + int8  # typical: INT4 model + INT8 KV
    fits = '✓' if total <= 80 else '✗'
    print(f'{seq_len:>10,} {bf16:>10.2f} {int8:>10.2f} {int4:>10.2f} '
          f'{total:>10.1f} GB {fits:>6}')

print(f'\n→ At 128K context, BF16 KV cache alone is {kv_cache_gb(80,8,128,131072):.0f} GB!')
print(f'→ INT8 KV cache is nearly lossless with per-channel quantization')
print(f'→ The combination of INT4 weights + INT8 KV is the standard for long-context serving')

---

## Exercise 19: Stochastic Rounding for Low-Precision Training ⭐⭐⭐

FP8 training on H100 uses stochastic rounding. Without it, small gradient updates are lost.

**Task**:
- (a) Implement `stochastic_round(x, resolution)` where resolution = ULP of the target format
- (b) Prove it's unbiased: E[SR(x)] = x (verify over 10K trials)
- (c) Simulate: accumulate gradient=0.001 into weight=1.0 with BF16 resolution (0.0078125)
  - Compare 10000 steps of deterministic vs stochastic rounding
  - The gradient is SMALLER than resolution/2 → deterministic rounding always drops it!

In [ ]:
# Exercise 19: Your Solution

print('Scaffold placeholder: replace the code below with your implementation and rerun this cell.')
print('The original scaffold is preserved below inside an if False block.')

if False:
    
    def deterministic_round(x: float, resolution: float) -> float:
        """Round to nearest multiple of resolution."""
        # YOUR CODE HERE
        pass
    
    def stochastic_round(x: float, resolution: float) -> float:
        """Stochastic rounding to nearest multiple of resolution.
        P(round up) = (x - floor)/resolution, P(round down) = 1 - P(round up)
        """
        # YOUR CODE HERE
        pass
    
    # (b) Verify unbiasedness
    # (c) Simulate gradient accumulation
    pass


In [ ]:
# Exercise 19: Solution ✅

def deterministic_round(x, resolution):
    return round(x / resolution) * resolution

def stochastic_round(x, resolution):
    lower = np.floor(x / resolution) * resolution
    upper = lower + resolution
    prob_up = (x - lower) / resolution
    return upper if np.random.random() < prob_up else lower

# (b) Unbiasedness test
np.random.seed(42)
res = 0.0078125  # BF16 ULP near 1.0
x_test = 1.003   # between two representable BF16 values
sr_results = [stochastic_round(x_test, res) for _ in range(10000)]
dr_result = deterministic_round(x_test, res)

print(f'(b) Unbiasedness test: x = {x_test}, resolution = {res}')
print(f'    E[SR(x)] = {np.mean(sr_results):.6f}  (should be ≈ {x_test})')
print(f'    RNE(x)   = {dr_result:.6f}  (biased toward {dr_result})')
print(f'    SR is unbiased: {np.isclose(np.mean(sr_results), x_test, atol=0.01)} ✓')

# (c) Gradient accumulation simulation
np.random.seed(42)
gradient = 0.001
n_steps = 10000

w_det = 1.0
for _ in range(n_steps):
    w_det = deterministic_round(w_det + gradient, res)

np.random.seed(42)
w_sr = 1.0
for _ in range(n_steps):
    w_sr = stochastic_round(w_sr + gradient, res)

expected = 1.0 + n_steps * gradient

print(f'\n(c) Gradient accumulation ({n_steps} steps, grad={gradient}, res={res}):')
print(f'    gradient/res = {gradient/res:.4f} (< 0.5 → deterministic ALWAYS rounds down!)')
print(f'    Expected:      {expected:.1f}')
print(f'    Deterministic: {w_det:.4f}  error={abs(w_det - expected):.4f}')
print(f'    Stochastic:    {w_sr:.4f}  error={abs(w_sr - expected):.4f}')
print(f'\n→ Deterministic rounding: weight NEVER changes because gradient rounds to zero!')
print(f'→ Stochastic rounding: gradient accumulates probabilistically → correct on average')
print(f'→ This is crucial for FP8 training on H100 — enables convergence at 8-bit')

---

## Exercise 20: Attention Logit Growth — Diagnosing Training Instability ⭐⭐⭐

Your 70B model's loss spikes after 50K steps. You suspect attention logit growth.

**Task** (requires PyTorch):
- (a) Simulate attention with Q, K of increasing norms (1× to 25×)
- (b) Show how max attention logit scales with Q/K norms
- (c) Show that softmax becomes "spiky" (max prob → 1.0) as logits grow
- (d) Implement QK-Norm: normalize Q, K before attention → bounded logits
- (e) Explain why this prevents NaN and gradient vanishing

In [ ]:
# Exercise 20: Your Solution

print('Scaffold placeholder: replace the code below with your implementation and rerun this cell.')
print('The original scaffold is preserved below inside an if False block.')

if False:
    
    if HAS_TORCH:
        d_k = 128  # head dimension
        seq_len = 64
    
        # (a)-(c): Simulate attention with growing Q, K norms
        for qk_scale in [1.0, 5.0, 10.0, 20.0, 25.0]:
            torch.manual_seed(42)
            Q = torch.randn(1, seq_len, d_k) * qk_scale
            K = torch.randn(1, seq_len, d_k) * qk_scale
    
            # YOUR CODE HERE: compute logits, softmax, entropy
            pass
    
        # (d) QK-Norm implementation
        # YOUR CODE HERE: normalize Q and K before computing attention
    else:
        print('PyTorch required for this exercise')
    pass


In [ ]:
# Exercise 20: Solution ✅

if HAS_TORCH:
    d_k = 128
    seq_len = 64

    # (a)-(c) Attention logit growth
    print('Attention Logit Growth — Simulating Training Instability')
    print('=' * 75)
    print(f'{"||Q||":>8} {"Max logit":>10} {"Max attn":>10} {"Entropy":>10} {"Status":>15}')
    print('-' * 60)

    for qk_scale in [1.0, 2.0, 5.0, 10.0, 15.0, 20.0, 25.0]:
        torch.manual_seed(42)
        Q = torch.randn(1, seq_len, d_k) * qk_scale
        K = torch.randn(1, seq_len, d_k) * qk_scale

        logits = torch.matmul(Q, K.transpose(-2, -1)) / np.sqrt(d_k)
        attn = F.softmax(logits, dim=-1)

        max_logit = logits.max().item()
        max_attn = attn.max().item()
        entropy = -(attn * torch.log(attn + 1e-10)).sum(-1).mean().item()

        if max_logit > 88:
            status = '💥 OVERFLOW'
        elif max_attn > 0.999:
            status = '⚠ Near-1hot'
        elif max_attn > 0.99:
            status = '⚠ Spiky'
        else:
            status = '✓ Safe'
        print(f'{qk_scale:>8.1f} {max_logit:>10.1f} {max_attn:>10.6f} {entropy:>10.2f} {status:>15}')

    # (d) QK-Norm fix
    print(f'\n--- QK-Norm Fix ---')
    print(f'{"||Q||_orig":>12} {"Max logit (no norm)":>20} {"Max logit (QK-Norm)":>22}')
    print('-' * 60)
    for qk_scale in [1.0, 10.0, 25.0]:
        torch.manual_seed(42)
        Q = torch.randn(1, seq_len, d_k) * qk_scale
        K = torch.randn(1, seq_len, d_k) * qk_scale

        # Without QK-Norm
        logits_raw = torch.matmul(Q, K.transpose(-2, -1)) / np.sqrt(d_k)

        # With QK-Norm: normalize Q and K to unit norm along feature dim
        Q_norm = F.normalize(Q, dim=-1)  # ||Q|| = 1 per token
        K_norm = F.normalize(K, dim=-1)  # ||K|| = 1 per token
        logits_normed = torch.matmul(Q_norm, K_norm.transpose(-2, -1)) / np.sqrt(d_k)

        print(f'{qk_scale:>12.1f} {logits_raw.max().item():>20.1f} '
              f'{logits_normed.max().item():>22.4f}')

    # (e) Explanation
    print(f'\n(e) Why QK-Norm works:')
    print(f'    After normalization: ||Q_i|| = ||K_j|| = 1')
    print(f'    Max logit = max(Q·K^T)/√d_k ≤ 1/√d_k = {1/np.sqrt(d_k):.4f}')
    print(f'    → Logits are BOUNDED regardless of training step')
    print(f'    → exp(logit) never overflows → no NaN')
    print(f'    → Attention stays well-distributed → healthy gradients')
    print(f'    → Used in Google\'s PaLM-2, LLaMA-3 (with logit capping)')
else:
    print('PyTorch required for this exercise')

---

## Summary: Skills Practiced

| # | Exercise | Core Skill | Level |
|---|----------|------------|-------|
| 1 | Base Conversion | Binary/hex representation | ⭐ |
| 2 | Two's Complement | Signed integer encoding (INT8/INT4) | ⭐ |
| 3 | IEEE 754 Decoder | Float bit inspection & debugging | ⭐ |
| 4 | Machine Epsilon | Precision limits, gradient loss | ⭐ |
| 5 | BF16 vs FP16 | Range/overflow analysis | ⭐ |
| 6 | Associativity Failure | Summation order, reproducibility | ⭐ |
| 7 | INT8 Quantization | Full quantize-dequantize pipeline | ⭐⭐ |
| 8 | Per-Channel Quant | Handling outlier channels | ⭐⭐ |
| 9 | INT4 Group Quant | W4A16 pipeline for inference | ⭐⭐ |
| 10 | NF4 (QLoRA) | Non-uniform quantization | ⭐⭐ |
| 11 | Catastrophic Cancellation | Stable variance, RMSNorm | ⭐⭐ |
| 12 | Kahan Summation | Compensated accumulation | ⭐⭐ |
| 13 | Stable Softmax | Max-subtraction trick | ⭐⭐ |
| 14 | Cross-Entropy/LSE | Log-sum-exp identity | ⭐⭐ |
| 15 | SQNR Analysis | Quantization noise measurement | ⭐⭐⭐ |
| 16 | Lloyd-Max Algorithm | Optimal non-uniform quantization | ⭐⭐⭐ |
| 17 | Memory Budget | GPU sizing for training/inference | ⭐⭐⭐ |
| 18 | KV Cache Memory | Long-context serving analysis | ⭐⭐⭐ |
| 19 | Stochastic Rounding | FP8 training enabler | ⭐⭐⭐ |
| 20 | Attention Logit Growth | Diagnosing training instability | ⭐⭐⭐ |

---

### References
- [notes.md](notes.md) — Complete 17-section mathematical reference
- [theory.ipynb](theory.ipynb) — Interactive demonstrations of all concepts

### Next Steps
- Continue to: **02-Sets-and-Logic**